# Knowledge Distillation on Google Colab

This notebook runs knowledge distillation experiments on Google Colab.

**Hardware:** A100 GPU recommended (T4/V100 also work)

**Quick Configs (~30-60 min):**
- BERT → DistilBERT on MNLI
- BERT → DistilBERT on QQP
- Llama 3B → 1B on translation

**Full Configs (~2-8 hours):**
- Use the non-quick versions for paper reproduction

## 1. Setup Environment

In [ ]:
# Mount Google Drive (saves checkpoints even if Colab disconnects)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Keep-alive script (prevents Colab from disconnecting due to inactivity)
from IPython.display import Javascript
display(Javascript('''
    setInterval(() => {
        console.log("Keep alive: " + new Date());
        document.querySelector("colab-connect-button")?.click();
    }, 60000);
'''))
print("Keep-alive script running (check browser console)")

In [ ]:
# Clone repository
!git clone https://github.com/CS-7150-DL/Distillation.git /content/Distillation
%cd /content/Distillation

In [ ]:
# Install dependencies
!pip install -q uv
!uv pip install -q -e . --system

## 2. Verify GPU

In [ ]:
!nvidia-smi

In [ ]:
from src.utils.environment import detect_environment, get_device

print(f"Environment: {detect_environment()}")
print(f"Device: {get_device()}")

## 3. Login to Services

In [ ]:
# W&B login (for experiment tracking)
# Get your API key at: https://wandb.ai/authorize
import wandb
wandb.login()

In [ ]:
# HuggingFace login (only needed for Llama models)
# Skip this cell if running BERT experiments
from huggingface_hub import notebook_login
notebook_login()

---
## 4. Run Training

Choose ONE of the following experiments to run.

### Option A: BERT → DistilBERT on MNLI (Recommended Start)

| Version | Time | Expected Accuracy |
|---------|------|-------------------|
| Quick | ~30-45 min | ~0.75-0.78 |
| Full | ~2-3 hours | ~0.817 |

In [ ]:
# MNLI - Quick version (~30-45 min)
!python train.py \
    --config configs/bert_distilbert_mnli_quick.yaml \
    --output_dir /content/drive/MyDrive/distillation/mnli_quick

In [ ]:
# MNLI - Full version (~2-3 hours)
!python train.py \
    --config configs/bert_distilbert_mnli.yaml \
    --output_dir /content/drive/MyDrive/distillation/mnli_full

### Option B: BERT → DistilBERT on QQP

| Version | Time | Expected Accuracy |
|---------|------|-------------------|
| Quick | ~30-45 min | ~0.85-0.88 |
| Full | ~2-3 hours | ~0.900 |

In [ ]:
# QQP - Quick version (~30-45 min)
!python train.py \
    --config configs/bert_distilbert_qqp_quick.yaml \
    --output_dir /content/drive/MyDrive/distillation/qqp_quick

In [ ]:
# QQP - Full version (~2-3 hours)
!python train.py \
    --config configs/bert_distilbert_qqp.yaml \
    --output_dir /content/drive/MyDrive/distillation/qqp_full

### Option C: Llama 3B → 1B Translation (Requires HF Login)

| Version | Time | Expected BLEU |
|---------|------|---------------|
| Quick | ~45-60 min | ~15-20 |
| Full | ~6-8 hours | ~27 |

In [ ]:
# Translation - Quick version (~45-60 min)
!python train.py \
    --config configs/llama_3b_1b_translation_quick.yaml \
    --output_dir /content/drive/MyDrive/distillation/translation_quick

In [ ]:
# Translation - Full version (~6-8 hours)
!python train.py \
    --config configs/llama_3b_1b_translation.yaml \
    --output_dir /content/drive/MyDrive/distillation/translation_full

---
## 5. View Results

In [ ]:
# View final metrics
import json
from pathlib import Path

# Change this to your output directory
output_dir = "/content/drive/MyDrive/distillation/mnli_quick"

trainer_state = Path(output_dir) / "trainer_state.json"

if trainer_state.exists():
    with open(trainer_state) as f:
        state = json.load(f)
    
    print("=" * 50)
    print("TRAINING RESULTS")
    print("=" * 50)
    
    if 'log_history' in state:
        # Get final eval metrics
        eval_logs = [log for log in state['log_history'] if 'eval_loss' in log]
        if eval_logs:
            final_eval = eval_logs[-1]
            print("\nFinal Evaluation Metrics:")
            for key, value in final_eval.items():
                if key.startswith('eval_'):
                    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
else:
    print(f"No trainer state found at {trainer_state}")
    print("Training may still be in progress or output_dir is incorrect.")

In [ ]:
# List saved checkpoints
!ls -la /content/drive/MyDrive/distillation/mnli_quick/

---
## 6. Download Model (Optional)

In [ ]:
# Download trained model to local machine
from google.colab import files
import shutil

output_dir = "/content/drive/MyDrive/distillation/mnli_quick"
shutil.make_archive("/content/trained_model", 'zip', output_dir)
files.download("/content/trained_model.zip")

---
## Troubleshooting

**Colab disconnects:**
- Make sure keep-alive cell ran
- On your laptop: run `caffeinate -dims` in terminal
- Checkpoints saved to Google Drive will persist

**Out of memory:**
- Restart runtime and try again
- Use quick configs (smaller batch sizes)

**W&B issues:**
- Set `WANDB_MODE=disabled` to skip W&B:
  ```python
  import os
  os.environ['WANDB_MODE'] = 'disabled'
  ```